In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

DATA_CLEANED_DIR = Path("../data/cleaned").resolve()
OUTPUTS_EDA = Path("../outputs/eda").resolve()
OUTPUTS_EDA.mkdir(parents=True, exist_ok=True)

print(f"Cleaned data directory: {DATA_CLEANED_DIR}")
print(f"Outputs EDA directory:  {OUTPUTS_EDA}")

In [ ]:
HOUSE_CONFIG = {
    1:  {"wm_col": "Appliance5", "occupancy": 2, "type": "Detached", "size": "4 bed", "model": "Beko WMC6140"},
    2:  {"wm_col": "Appliance2", "occupancy": 4, "type": "Semi-detached", "size": "3 bed", "model": "LG F1289TD"},
    3:  {"wm_col": "Appliance6", "occupancy": 2, "type": "Detached", "size": "3 bed", "model": "Standard UK Washer"},
    20: {"wm_col": "Appliance4", "occupancy": 2, "type": "Detached", "size": "3 bed", "model": "Standard UK Washer"}
}

def load_resample_household(house_id, nrows=400000):
    filepath = DATA_CLEANED_DIR / f"CLEAN_House{house_id}.csv"
    assert filepath.exists(), f"File {filepath} not found!"
    cfg = HOUSE_CONFIG[house_id]
    wm_col = cfg["wm_col"]
    
    print(f"Loading House {house_id} ({nrows:,} rows)...")
    cols = ['Time', 'Aggregate', wm_col]
    df = pd.read_csv(filepath, nrows=nrows, usecols=cols)
    df['DateTime'] = pd.to_datetime(df['Time'], errors='coerce')
    df = df.dropna(subset=['DateTime']).sort_values('DateTime')
    
    df['Aggregate'] = pd.to_numeric(df['Aggregate'], errors='coerce').clip(lower=0, upper=15000)
    df['Washing_Machine'] = pd.to_numeric(df[wm_col], errors='coerce').clip(lower=0, upper=4000)
    
    df_1m = df.set_index('DateTime')[['Aggregate', 'Washing_Machine']].resample('1min').mean()
    df_1m = df_1m.interpolate(method='time', limit=3, limit_direction='forward')
    df_1m['is_outage'] = df_1m['Aggregate'].isna()
    return df_1m

house_data = {}
for hid in [1, 2, 3, 20]:
    house_data[hid] = load_resample_household(hid, nrows=400000)
    print(f"  -> House {hid}: {len(house_data[hid]):,} mins from {house_data[hid].index.min().date()} to {house_data[hid].index.max().date()}")

In [ ]:
quality_records = []
for hid, df_h in house_data.items():
    cfg = HOUSE_CONFIG[hid]
    valid_wm = df_h['Washing_Machine'].dropna()
    total_bins = len(df_h)
    outages = df_h['is_outage'].sum()
    off_bins = (valid_wm <= 5.0).sum()
    active_bins = (valid_wm > 15.0).sum()
    
    quality_records.append({
        "House": f"House {hid}",
        "Occupants": cfg["occupancy"],
        "Dwelling": cfg["type"],
        "Start Date": df_h.index.min().strftime("%Y-%m-%d"),
        "End Date": df_h.index.max().strftime("%Y-%m-%d"),
        "Total Days": round((df_h.index.max() - df_h.index.min()).total_seconds() / 86400, 1),
        "Total 1-Min Bins": total_bins,
        "Missing / Outage (%)": round(100.0 * outages / total_bins, 2),
        "Standby <=5W (%)": round(100.0 * off_bins / len(valid_wm), 2),
        "Active >15W Bins": active_bins,
        "Active Time (%)": round(100.0 * active_bins / len(valid_wm), 2)
    })

df_quality = pd.DataFrame(quality_records)
df_quality

In [ ]:
def detect_washing_cycles(series_power, active_thresh=15.0, max_pause_mins=3, min_duration_mins=15, min_peak_power=500.0, min_energy_kwh=0.05):
    s = series_power.fillna(0)
    is_active = (s >= active_thresh).astype(int)
    
    padded = is_active.copy()
    change = (padded != padded.shift(1)).cumsum()
    groups = padded.groupby(change)
    for _, grp in groups:
        if grp.iloc[0] == 0 and len(grp) <= max_pause_mins:
            padded.loc[grp.index] = 1
            
    block_id = (padded != padded.shift(1)).cumsum()
    cycles = []
    for bid, grp in padded.groupby(block_id):
        if grp.iloc[0] == 1:
            start_t, end_t = grp.index[0], grp.index[-1]
            dur = len(grp)
            if dur >= min_duration_mins:
                cycle_p = s.loc[start_t:end_t]
                peak_p = float(cycle_p.max())
                avg_p = float(cycle_p.mean())
                energy_kwh = float(cycle_p.sum() / 60.0 / 1000.0)
                if peak_p >= min_peak_power or energy_kwh >= min_energy_kwh:
                    cycles.append({
                        "start_time": start_t, "end_time": end_t,
                        "duration_mins": dur, "peak_power_w": round(peak_p, 1),
                        "avg_power_w": round(avg_p, 1), "energy_kwh": round(energy_kwh, 4),
                        "hour": start_t.hour, "day_of_week": start_t.strftime("%A"),
                        "weekday_idx": start_t.weekday()
                    })
    return pd.DataFrame(cycles)

house_cycles = {}
for hid, df_h in house_data.items():
    cdf = detect_washing_cycles(df_h['Washing_Machine'])
    cdf['house_id'] = hid
    house_cycles[hid] = cdf
    print(f"House {hid}: Detected {len(cdf)} active washing cycles.")

In [ ]:
df_all_cycles = pd.concat(list(house_cycles.values()), ignore_index=True)
cycle_comp = []
for hid, cdf in house_cycles.items():
    cfg = HOUSE_CONFIG[hid]
    days = (house_data[hid].index.max() - house_data[hid].index.min()).total_seconds() / 86400
    weeks = days / 7.0
    cycle_comp.append({
        "House": f"House {hid}",
        "Occupants": cfg["occupancy"],
        "Cycles Count": len(cdf),
        "Cycles / Week": round(len(cdf) / weeks, 2),
        "Median Duration (min)": round(cdf["duration_mins"].median(), 1),
        "Mean Duration (min)": round(cdf["duration_mins"].mean(), 1),
        "Median Peak Power (W)": round(cdf["peak_power_w"].median(), 1),
        "Mean Energy (kWh/cycle)": round(cdf["energy_kwh"].mean(), 3),
        "Total Energy (kWh)": round(cdf["energy_kwh"].sum(), 2)
    })
df_cycle_summary = pd.DataFrame(cycle_comp)
df_cycle_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 1. Hourly Pattern
sns.histplot(data=df_all_cycles, x="hour", hue="house_id", multiple="dodge", discrete=True, palette="Set2", shrink=0.8, ax=axes[0])
axes[0].set_title("Washing Machine Starts by Hour of Day (0–23 h)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Hour of Day (UTC)")
axes[0].set_ylabel("Cycle Count")
axes[0].set_xticks(range(0, 24, 2))
axes[0].grid(True, linestyle="--", alpha=0.5)

# 2. Weekly Pattern
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
sns.countplot(data=df_all_cycles, x="day_of_week", hue="house_id", order=day_order, palette="Set2", ax=axes[1])
axes[1].set_title("Washing Machine Starts by Day of Week", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Day of Week")
axes[1].set_ylabel("Cycle Count")
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
fig_time_path = OUTPUTS_EDA / "wm_diurnal_weekly_patterns.png"
fig.savefig(fig_time_path, dpi=200, bbox_inches="tight")
print(f"Saved temporal patterns figure to {fig_time_path}")
plt.show()

In [ ]:
trace_start = "2013-10-13 00:00:00"
trace_end   = "2013-10-14 00:00:00"
df_h1_24h = house_data[1].loc[trace_start:trace_end]

fig, ax = plt.subplots(figsize=(14, 5.5))
ax.plot(df_h1_24h.index, df_h1_24h['Aggregate'], color='#1f77b4', lw=1.2, label='Whole-House Aggregate Power')
ax.plot(df_h1_24h.index, df_h1_24h['Washing_Machine'], color='#d62728', lw=1.8, label='Washing Machine [Beko WMC6140]')
ax.fill_between(df_h1_24h.index, 0, df_h1_24h['Washing_Machine'], color='#d62728', alpha=0.25)

ax.set_title("Visualisation (i): Representative 24-Hour Electricity Profile: House 1 (Oct 13, 2013)", fontsize=13, fontweight='bold')
ax.set_ylabel("Active Power (Watts)", fontsize=11)
ax.set_xlabel("Time of Day (UTC)", fontsize=11)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.legend(loc='upper right', frameon=True, fontsize=10)
ax.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
fig1_path = OUTPUTS_EDA / "vis_1_representative_24h_trace.png"
fig.savefig(fig1_path, dpi=200, bbox_inches="tight")
print(f"Saved Vis (i) to {fig1_path}")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# House 1 Cycle
c1 = house_cycles[1].iloc[0]
s1 = house_data[1].loc[c1['start_time'] - pd.Timedelta(minutes=10):c1['end_time'] + pd.Timedelta(minutes=10), 'Washing_Machine']
axes[0].plot(s1.index, s1.values, color='#d62728', lw=1.8, marker='.', markersize=4)
axes[0].axvspan(c1['start_time'], c1['end_time'], color='green', alpha=0.15, label=f"Detected Cycle ({c1['duration_mins']}m, {c1['energy_kwh']:.2f}kWh)")
axes[0].set_title(f"House 1 (Beko): Detected Cycle\nPeak: {c1['peak_power_w']} W, Duration: {c1['duration_mins']} mins", fontsize=11, fontweight='bold')
axes[0].set_ylabel("Power (Watts)")
axes[0].set_xlabel("Time (UTC)")
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[0].legend(loc='upper right', fontsize=9)
axes[0].grid(True, linestyle="--", alpha=0.5)

# House 2 Cycle
c2 = house_cycles[2].iloc[1]
s2 = house_data[2].loc[c2['start_time'] - pd.Timedelta(minutes=10):c2['end_time'] + pd.Timedelta(minutes=10), 'Washing_Machine']
axes[1].plot(s2.index, s2.values, color='#8c564b', lw=1.8, marker='.', markersize=4)
axes[1].axvspan(c2['start_time'], c2['end_time'], color='green', alpha=0.15, label=f"Detected Cycle ({c2['duration_mins']}m, {c2['energy_kwh']:.2f}kWh)")
axes[1].set_title(f"House 2 (LG): Detected Cycle\nPeak: {c2['peak_power_w']} W, Duration: {c2['duration_mins']} mins", fontsize=11, fontweight='bold')
axes[1].set_ylabel("Power (Watts)")
axes[1].set_xlabel("Time (UTC)")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
fig2_path = OUTPUTS_EDA / "vis_2_detected_cycles_zoomed.png"
fig.savefig(fig2_path, dpi=200, bbox_inches="tight")
print(f"Saved Vis (ii) to {fig2_path}")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.boxplot(data=df_all_cycles, x='house_id', y='duration_mins', palette='Set2', ax=axes[0])
axes[0].set_title("Cycle Duration by Household", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Household ID")
axes[0].set_ylabel("Duration (Minutes)")
axes[0].grid(True, linestyle="--", alpha=0.5)

sns.boxplot(data=df_all_cycles, x='house_id', y='peak_power_w', palette='Set2', ax=axes[1])
axes[1].set_title("Peak Power by Household", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Household ID")
axes[1].set_ylabel("Peak Power (Watts)")
axes[1].grid(True, linestyle="--", alpha=0.5)

sns.boxplot(data=df_all_cycles, x='house_id', y='energy_kwh', palette='Set2', ax=axes[2])
axes[2].set_title("Energy Consumption per Cycle", fontsize=12, fontweight='bold')
axes[2].set_xlabel("Household ID")
axes[2].set_ylabel("Energy (kWh)")
axes[2].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
fig3_path = OUTPUTS_EDA / "vis_3_cross_house_comparison.png"
fig.savefig(fig3_path, dpi=200, bbox_inches="tight")
print(f"Saved Vis (iii) to {fig3_path}")
plt.show()